# **BRONZE LAYER**

In [0]:
from pyspark.sql.functions import current_timestamp, date_format

#if want to test code else where change these vars for yourseld
catalog_name = "26100355-pa2" 
schema_name = "bronze"
volume_name = "temp"
base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
#made vol in bronze layer for me
landing_zone = f"{base_path}/flight_landing_zone"
schema_location = f"{base_path}/schemas/bronze_flights"
checkpoint_path = f"{base_path}/checkpoints/bronze_flights"

print(f"Copying data")
dbutils.fs.cp("dbfs:/databricks-datasets/flights/departuredelays.csv", landing_zone)
print("Start Auto-Loader read")
raw_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true") 
    .load(landing_zone))
bronze_stream = raw_stream.withColumn(
    "ingestion_timestamp",
    date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
)


print(f"Writing")
write_query = (bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`{schema_name}`.`flights`"))
write_query.awaitTermination()
print("Bronze done")

Copying data
Start Auto-Loader read
Writing
Bronze done




1.
* its a setting in Databricks Auto Loader that controls how the pipeline reacts when the incoming data's structure changes .
* Its used when ingesting raw data from external sources where you don’t control the schema

2.
Triggers determine exactly when and how often Spark processes a new batch of streaming data. 
* Default: Processes the next batch immediately for the lowest possible latency.
* Fixed-Interval: Processes at a specific time interval for predictable cluster usage.
* AvailableNow: Processes all currently available data and then shuts the stream down. This is used to save costs by running streams on a schedule instead of 24/7.
* Continuous: Processes data row-by-row for ultra-low latency.

# **SILVER LAYER**

In [0]:
from pyspark.sql.functions import col, concat, lit, to_timestamp, date_format

catalog_name = "26100355-pa2"
bronze_table = f"`{catalog_name}`.`bronze`.`flights`"
bronze_stream = spark.readStream.table(bronze_table)

silver_df = (bronze_stream
    .withColumn("delay", col("delay").cast("integer"))
    .withColumn("distance", col("distance").cast("integer"))
    .withColumn("origin", col("origin").cast("string"))
    .withColumn("destination", col("destination").cast("string"))
    .withColumn("raw_date_with_year", concat(lit("2025"), col("date")))
    .withColumn("event_timestamp", to_timestamp(col("raw_date_with_year"), "yyyyMMddHHmm"))
    .withColumn("date", date_format(col("event_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .drop("raw_date_with_year")
)